## QUESTÃO 1 - EDA

Premissas obrigatórias:<small>
- Utilizar apenas o dataset vendas_2023_2024.csv
- Não realizar limpeza ou tratamento dos dados
- Apenas observar, agregar e descrever as informações</small>

Tarefas:
<small>
1. Visão geral do dataset:

- Quantidade total de linhas  
- Quantidade total de colunas  
- Intervalo de datas (datas mínima e máxima):

2. Análise de valores numéricos. Para a coluna total, calcular:
- Valor mínimo  
- Valor máximo  
- Valor médio  

3. Interpretação:
Apresentar breve diagnóstico sobre a confiabilidade do dataset vendas_2023_2024.csv para análises futuras, destacando:
- Possíveis outliers na coluna total  
- Qualidade dos dados:
  - Presença de valores nulos  
  - Inconsistências identificadas  
- Avaliação geral:
  - O dataset está pronto para análise ou exige tratamento prévio?
 </small>
 ---

In [34]:
# IMPORTANDO BIBLIOTECAS
import pandas as pd 
from datetime import datetime

In [35]:
# CARREGANDO O DATASET
df = pd.read_csv('../data/raw/vendas_2023_2024.csv')
print(f"Linhas: {df.shape[0]} Colunas: {df.shape[1]}")

Linhas: 9895 Colunas: 6


In [36]:
# CONFERINDO AS DATAS
df_data = df.copy()
df_data['sale_date'] = pd.to_datetime(df['sale_date'], format='mixed', dayfirst=True)
print(f"Data mínima: {df_data['sale_date'].min().strftime('%d-%m-%y')} e Data máxima: {df_data['sale_date'].max().strftime('%d-%m-%y')}")

Data mínima: 01-01-23 e Data máxima: 31-12-24


In [37]:
#ANÁLISE DA COLUNA TOTAL
print(f"Valor mínimo: {df['total'].min()}")
print(f"Valor máximo: {df['total'].max()}")
print(f"Valor médio: {round(df['total'].mean(), 2)}")
print(f"Mediana: {df['total'].median()}")

Valor mínimo: 294.5
Valor máximo: 2222973.0
Valor médio: 263797.83
Mediana: 82225.0


In [38]:
#AVALIANDO OUTLIERS PELO MÉTODO INTERQUARTIL
q1 = df['total'].quantile(0.25) 
q3 = df['total'].quantile(0.75)
iqr = q3 - q1
lim_inferior = q1 - 1.5 * iqr
lim_superior = q3 + 1.5 * iqr

outliers_inferiores = df[df['total'] < lim_inferior]
outliers_superiores = df[df['total'] > lim_superior]

print(f"Q1: {q1}\nQ3: {q3}")
print(f"limite inferior: {lim_inferior}\nlimite superior: {lim_superior}")

print(f"outliers inferiores: {outliers_inferiores.shape[0]}")
print(f"outliers superiores: {outliers_superiores.shape[0]}")

Q1: 23138.2
Q3: 339094.5
limite inferior: -450796.24999999994
limite superior: 813028.95
outliers inferiores: 0
outliers superiores: 1018


In [39]:
# RANKING DE OUTLIERS
outliers_ranking = outliers_superiores.sort_values(by='total', ascending=False)
top_outliers = outliers_ranking[['id_product', 'total']].head(10)

print("Top 10 maiores outliers de vendas:")
display(top_outliers)

Top 10 maiores outliers de vendas:


,id_product,total
8905,76,2222973.00
3873,76,2222973.00
9623,76,2222973.00
1578,76,2222973.00
7930,73,2147399.00
8856,76,2111824.35
1974,76,2111824.35
8800,76,2074775.00
5014,76,2074775.00
5183,97,2030026.00


Os outliers significativos são os superiores com 1018 casos (cerca de 10% dos dados), que indicam valores  elevados acima dos 2 milhões, principalmente quandoo comparados às medidas de tendencia central deste dataframe. A média de 263 mil e a mediana 82 mil evidenciam a influência dos extremos de cima.

Além disso, só de observar o ranking dos 10 maiores outliers, percebemos mesmo produto (ID 76) aparecendo multiplas vezes entre os maiores, sendo que em quatro delas com o mesmo valor máximo de 2222973, o que pode nos indicar alguma anomalia, como talvez, registros duplicados.

In [40]:
#ANALISANDO OUTLIERS SUSPEITOS
outliers_suspeitos = df[(df['id_product'] == 76) & (df['total'] > 2222900)]
display(outliers_suspeitos.head())

,id,id_client,id_product,qtd,total,sale_date
1578,1598,9,76,15,2222973.0,2024-07-19
3873,3911,39,76,15,2222973.0,07-08-2023
8905,9000,47,76,15,2222973.0,2023-04-30
9623,9727,22,76,15,2222973.0,2024-07-08


Apesar da possibilidade de comportamento anômalo, com o mesmo produto registrando o mesmo total elevado quatro vezes, podemos visualizar que não se trata de duplicatas, apenas compras do mesmo produto em mesma quantidade, realizadas por clientes diferentes em datas distintas.

In [41]:
#ANALISANDO VALORES NULOS, NEGATIVOS E DUPLICADOS
valores_nulos = df.isnull().sum()
print(f"Valores nulos: {valores_nulos}")
valores_negativos = (df.select_dtypes(include='number') < 0).sum()
print(f"Valores negativos: {valores_negativos}")
valores_duplicados = df.duplicated().sum()
print(f"Valores duplicados: {valores_duplicados}")

Valores nulos: id            0
id_client     0
id_product    0
qtd           0
total         0
sale_date     0
dtype: int64
Valores negativos: id            0
id_client     0
id_product    0
qtd           0
total         0
dtype: int64
Valores duplicados: 0


In [42]:
#ANALISANDO FORMATOS DA COLUNA sale_date
df['sale_date'].astype(str).str.replace(r'\d', 'X', regex=True).value_counts()

sale_date
XX-XX-XXXX    4982
XXXX-XX-XX    4913
Name: count, dtype: int64

Os dados são de qualidade, não constando valores nulos, negativos ou duplicados. Entretanto, temos inconsistências no formato padrão de datas da coluna "sale_date", que apresenta dois tipos de configuração. 

Ademais, podemos considerar o dataset parcialmente apto para prosseguir nas análises, e que estará completamente pronto após solucionadas inconsistências.